In [0]:


# COMMAND ----------
# 1. IMPORTS

from pyspark.sql import functions as F
from pyspark.sql.window import Window

import uuid


# COMMAND ----------
# 2. CONFIGURATION

HISTORICAL_TABLE = (
    "fraud_mlops.gold.fraud_training_snapshot"
)

PRODUCTION_TABLE = (
    "fraud_mlops.gold.production_drift_window"
)

GROUND_TRUTH_TABLE = (
    "fraud_mlops.gold.production_ground_truth"
)

OUTPUT_TABLE = (
    "fraud_mlops.gold.fraud_retraining_snapshot_v3"
)


FEATURES = [
    "amount",
    "log_amount",
    "type_cash_in",
    "type_cash_out",
    "type_debit",
    "type_payment",
    "type_transfer",
    "origin_txn_count_prior",
    "origin_avg_amount_prior",
    "origin_total_amount_prior",
    "amount_vs_origin_avg",
    "origin_has_history",
]


# New matured production split
TRAIN_RATIO = 0.70
VALIDATION_RATIO = 0.15
TEST_RATIO = 0.15


# COMMAND ----------
# 3. LOAD HISTORICAL LABELED DATA

historical_df = (
    spark.table(HISTORICAL_TABLE)

    .select(
        "transaction_id",
        "step",
        "hour_of_day",
        "day_number",
        *FEATURES,
        "is_fraud",
        "dataset_split"
    )

    .filter(
        F.col("is_fraud").isNotNull()
    )

    .withColumn(
        "data_source",
        F.lit("historical")
    )
)


historical_count = historical_df.count()

print(
    "Historical labeled rows:",
    historical_count
)


# COMMAND ----------
# 4. LOAD MATURED PRODUCTION LABELS

production_df = (
    spark.table(PRODUCTION_TABLE)
    .alias("p")

    .join(
        spark.table(GROUND_TRUTH_TABLE)
        .filter(
            F.col("label_status") == "AVAILABLE"
        )
        .alias("g"),

        F.col("p.transaction_id")
        == F.col("g.transaction_id"),

        "inner"
    )

    .select(
        F.col("p.transaction_id"),
        F.col("p.step"),
        F.col("p.hour_of_day"),
        F.col("p.day_number"),

        *[
            F.col(f"p.{feature}")
            for feature in FEATURES
        ],

        F.col("g.is_fraud")
    )

    .withColumn(
        "data_source",
        F.lit("production")
    )
)


production_count = production_df.count()

print(
    "Matured production rows:",
    production_count
)


if production_count == 0:
    raise ValueError(
        "No matured production labels available."
    )


# COMMAND ----------
# 5. CREATE FRESH TIME-BASED SPLIT
#    FOR NEW PRODUCTION DATA

# Earlier simulated transaction steps first,
# later transaction steps last.

production_window = Window.orderBy(
    F.col("step"),
    F.col("transaction_id")
)


production_df = (
    production_df
    .withColumn(
        "_row_number",
        F.row_number().over(
            production_window
        )
    )
)


train_end = int(
    production_count * TRAIN_RATIO
)

validation_end = int(
    production_count
    * (TRAIN_RATIO + VALIDATION_RATIO)
)


production_df = (
    production_df

    .withColumn(
        "dataset_split",

        F.when(
            F.col("_row_number") <= train_end,
            F.lit("train")
        )

        .when(
            F.col("_row_number") <= validation_end,
            F.lit("validation")
        )

        .otherwise(
            F.lit("test")
        )
    )

    .drop("_row_number")
)


# COMMAND ----------
# 6. VERIFY NEW PRODUCTION SPLIT

production_split_summary = (
    production_df

    .groupBy(
        "dataset_split"
    )

    .agg(
        F.count("*").alias("rows"),
        F.sum("is_fraud").alias("fraud_rows")
    )

    .orderBy(
        "dataset_split"
    )
)


display(
    production_split_summary
)


# COMMAND ----------
# 7. COMBINE HISTORICAL + NEW PRODUCTION DATA

retraining_df = (
    historical_df

    .unionByName(
        production_df
    )
)


# COMMAND ----------
# 8. ADD SNAPSHOT METADATA

snapshot_id = str(
    uuid.uuid4()
)


retraining_df = (
    retraining_df

    .withColumn(
        "retraining_snapshot_id",
        F.lit(snapshot_id)
    )

    .withColumn(
        "snapshot_created_at",
        F.current_timestamp()
    )
)


# COMMAND ----------
# 9. VERIFY FINAL RETRAINING DATASET

summary_df = (
    retraining_df

    .groupBy(
        "data_source",
        "dataset_split"
    )

    .agg(
        F.count("*").alias("rows"),
        F.sum("is_fraud").alias("fraud_rows")
    )

    .orderBy(
        "data_source",
        "dataset_split"
    )
)


display(
    summary_df
)


total_rows = retraining_df.count()

print(
    "Total retraining rows:",
    total_rows
)

print(
    "Snapshot ID:",
    snapshot_id
)


# COMMAND ----------
# 10. SAVE IMMUTABLE RETRAINING SNAPSHOT

(
    retraining_df
    .write
    .format("delta")
    .mode("error")
    .saveAsTable(
        OUTPUT_TABLE
    )
)


# COMMAND ----------
# 11. FINAL VERIFICATION

print(
    "Saved rows:",
    spark.table(
        OUTPUT_TABLE
    ).count()
)


display(
    spark.table(
        OUTPUT_TABLE
    )

    .groupBy(
        "dataset_split",
        "data_source"
    )

    .agg(
        F.count("*").alias("rows"),
        F.sum("is_fraud").alias("fraud_rows")
    )

    .orderBy(
        "dataset_split",
        "data_source"
    )
)


# COMMAND ----------
# 12. COMPLETION MESSAGE

print(
    f"Retraining snapshot created successfully.\n"
    f"Snapshot ID: {snapshot_id}\n"
    f"Historical rows: {historical_count}\n"
    f"Matured production rows: {production_count}\n"
    f"Total rows: {total_rows}\n"
    f"Output table: {OUTPUT_TABLE}"
)